# APEX Go2 + MuJoCo Warp + SNN Actor

APEX公式Go2のモーションCSV、45D/77D観測、DecAP、2-Critic PPOを使います。ActorだけSNNに替え、元のSNNサイズ（encoder 64、hidden 256/256、decoder 256）へ戻します。Isaac GymとMuJoCo Warpの物理差、およびSNN内部4 tickは論文からの相違点です。

CUDA版PyTorchと `mujoco-warp==3.14.0` 等のrequirementsが必要です。必要なら先に `pip install mujoco-warp==3.14.0 pandas pyyaml wandb tensorboard imageio[ffmpeg]` を実行してください。このNotebookは同じフォルダにある既存の `APEX` を参照します。APEXのcloneやcheckoutは行いません。下のソースセルはSNN用ファイルを作成します。


In [ ]:
from pathlib import Path
import os, sys, subprocess
APEX_ROOT = Path(os.environ.get('APEX_ROOT', './APEX')).expanduser().resolve()
if not (APEX_ROOT / 'legged_gym/envs/param_config.yaml').is_file():
    raise FileNotFoundError(f'Existing APEX checkout missing: {APEX_ROOT}')
print('Existing APEX checkout:', APEX_ROOT)


## env_snn.py


In [ ]:
%%writefile env_snn.py
"""Go2 APEX flat-terrain VecEnv backed by MuJoCo Warp.

Keep the original APEX rsl_rl package as the learner. This module replaces
only the Isaac Gym environment for the repository's default Go2 configuration.
"""
from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd
import torch
import mujoco
import mujoco_warp as mjw
import warp as wp
import xml.etree.ElementTree as ET

from go2_mjcf import build_go2_mjcf

LEG_ORDER = ("FL", "FR", "RL", "RR")
JOINTS = tuple(f"{leg}_{part}_joint" for leg in LEG_ORDER for part in ("hip", "thigh", "calf"))
FEET = tuple(f"{leg}_foot" for leg in LEG_ORDER)


def quat_rotate_inverse_xyzw(q: torch.Tensor, v: torch.Tensor) -> torch.Tensor:
    xyz, w = q[..., :3], q[..., 3:4]
    return v * (2 * w.square() - 1) - 2 * w * torch.cross(xyz, v, dim=-1) + 2 * xyz * (xyz * v).sum(-1, keepdim=True)


def yaw_rotate_inverse(q: torch.Tensor, v: torch.Tensor) -> torch.Tensor:
    yaw = torch.atan2(2 * (q[:, 3] * q[:, 2] + q[:, 0] * q[:, 1]),
                      1 - 2 * (q[:, 1].square() + q[:, 2].square()))
    c, s = torch.cos(yaw)[:, None], torch.sin(yaw)[:, None]
    return torch.stack((c * v[..., 0] + s * v[..., 1],
                        -s * v[..., 0] + c * v[..., 1], v[..., 2]), dim=-1)


class ApexGo2Warp:
    num_obs = 45
    num_privileged_obs = 77
    num_actions = 12
    dt = 0.02
    decimation = 4

    def __init__(self, apex_root: str | Path, num_envs: int = 256, device: str = "cuda:0",
                 motion: str = "imitation_data/animal_mocap/go2_retarget_canter_2ms.csv",
                 add_noise: bool = True, enable_prior: bool = True,
                 domain_randomization: bool = True, resample_commands: bool = True,
                 push_robots: bool = True, auto_reset: bool = True):
        if not torch.cuda.is_available():
            raise RuntimeError("MuJoCo Warp training requires an NVIDIA CUDA GPU")
        self.device = torch.device(device)
        self.add_noise = add_noise
        self.enable_prior = enable_prior
        self.domain_randomization = domain_randomization
        self.resample_commands = resample_commands
        self.push_robots = push_robots
        self.auto_reset = auto_reset
        wp.init()
        wp.set_device(str(self.device))
        self.apex_root = Path(apex_root).resolve()
        urdf = self.apex_root / "resources/robots/go2/urdf/go2.urdf"
        self.cpu_model = mujoco.MjModel.from_xml_string(build_go2_mjcf(urdf))
        self.model = mjw.put_model(self.cpu_model)
        self.data = mjw.make_data(self.cpu_model, nworld=num_envs)
        self.qpos = wp.to_torch(self.data.qpos)
        self.qvel = wp.to_torch(self.data.qvel)
        self.ctrl = wp.to_torch(self.data.ctrl)
        self.num_envs = num_envs
        self.joint_qpos = [int(self.cpu_model.jnt_qposadr[self.cpu_model.joint(n).id]) for n in JOINTS]
        self.joint_qvel = [int(self.cpu_model.jnt_dofadr[self.cpu_model.joint(n).id]) for n in JOINTS]
        self.motor_ids = [self.cpu_model.actuator(n).id for n in JOINTS]
        self.foot_ids = [self.cpu_model.body(n).id for n in FEET]
        self.foot_root_ids = [int(self.cpu_model.body_rootid[i]) for i in self.foot_ids]
        self.penalty_ids = [i for i in range(1, self.cpu_model.nbody)
                            if any(s in self.cpu_model.body(i).name for s in ("base", "hip", "thigh", "calf", "trunk"))]
        self.termination_ids = [i for i in range(1, self.cpu_model.nbody)
                                if any(s in self.cpu_model.body(i).name for s in ("base", "hip"))]
        urdf_root = ET.parse(urdf).getroot()
        effort = {j.get("name"): float(j.find("limit").get("effort"))
                  for j in urdf_root.findall("joint") if j.get("type") == "revolute"}
        self.torque_limits = torch.tensor([effort[n] for n in JOINTS], device=self.device)
        default = {"hip": (0.1, -0.1, 0.1, -0.1),
                   "thigh": (0.8, 0.8, 1.0, 1.0), "calf": (-1.5,) * 4}
        self.default_dof_pos = torch.tensor([default[part][i] for i in range(4)
                                             for part in ("hip", "thigh", "calf")], device=self.device)
        self.p_gains = torch.full((12,), 20.0, device=self.device)
        self.d_gains = torch.full((12,), 0.5, device=self.device)
        self.actions = torch.zeros(num_envs, 12, device=self.device)
        self.last_actions = torch.zeros_like(self.actions)
        self.last_dof_vel = torch.zeros_like(self.actions)
        self.torques = torch.zeros_like(self.actions)
        self.commands = torch.zeros(num_envs, 4, device=self.device)
        self.commands_scale = torch.tensor([2.0, 2.0, 0.25], device=self.device)
        self.motor_offsets = torch.zeros_like(self.actions)
        self.Kp_factors = torch.ones(num_envs, 1, device=self.device)
        self.Kd_factors = torch.ones(num_envs, 1, device=self.device)
        self.motor_strengths = torch.ones(num_envs, 1, device=self.device)
        self.decap_factor = torch.ones(num_envs, 1, device=self.device)
        self.episode_length_buf = torch.zeros(num_envs, dtype=torch.long, device=self.device)
        self.imitation_index = torch.zeros(num_envs, dtype=torch.long, device=self.device)
        self.reset_buf = torch.zeros(num_envs, dtype=torch.bool, device=self.device)
        self.time_out_buf = torch.zeros_like(self.reset_buf)
        self.last_foot_velocities = torch.zeros(num_envs, 4, 3, device=self.device)
        self.last_contacts = torch.zeros(num_envs, 4, dtype=torch.bool, device=self.device)
        self.global_training_iteration = 0
        self.torque_ref_decay_factor = 0
        self.common_step_counter = 0
        self.motion = torch.tensor(pd.read_csv(self.apex_root / motion).to_numpy(dtype=np.float32), device=self.device)
        if self.motion.shape[1] < 40:
            raise ValueError("Go2 APEX imitation CSV needs at least 40 columns")
        self.max_episode_length = self.motion.shape[0] - 1
        self.max_episode_length_s = self.max_episode_length * self.dt
        self.obs_buf = torch.zeros(num_envs, 45, device=self.device)
        self.privileged_obs_buf = torch.zeros(num_envs, 77, device=self.device)
        self.rew_buf = torch.zeros(num_envs, 2, device=self.device)
        self.extras = {}
        self.reset()
        # The standing target is calculated once from the default pose.
        self.default_foot_pos_body_frame = yaw_rotate_inverse(self.base_quat,
             self.foot_positions - self.base_pos[:, None, :]).reshape(num_envs, 12).clone()

    @property
    def dof_pos(self):
        return self.qpos[:, self.joint_qpos]

    @property
    def dof_vel(self):
        return self.qvel[:, self.joint_qvel]

    def _refresh(self):
        self.base_pos = self.qpos[:, :3]
        self.base_quat = torch.cat((self.qpos[:, 4:7], self.qpos[:, 3:4]), dim=1)
        self.base_lin_vel = quat_rotate_inverse_xyzw(self.base_quat, self.qvel[:, :3])
        # MuJoCo free-joint qvel stores angular velocity in the body frame.
        self.base_ang_vel = self.qvel[:, 3:6]
        self.projected_gravity = quat_rotate_inverse_xyzw(self.base_quat,
            torch.tensor([0., 0., -1.], device=self.device).expand(self.num_envs, 3))
        self.foot_positions = wp.to_torch(self.data.xpos)[:, self.foot_ids, :]
        # cvel uses the root subtree COM as its reference point and stores
        # (angular, linear) world velocities; shift to each foot body origin.
        cvel = wp.to_torch(self.data.cvel)[:, self.foot_ids, :]
        subtree_com = wp.to_torch(self.data.subtree_com)[:, self.foot_root_ids, :]
        self.foot_velocities = cvel[..., 3:6] + torch.cross(
            cvel[..., :3], self.foot_positions - subtree_com, dim=-1)
        self.contact_forces = wp.to_torch(self.data.cfrc_ext)[..., 3:6]

    def _sample_commands(self, ids):
        if ids.numel() == 0:
            return
        ref = self.motion[self.imitation_index[ids].clamp(0, self.max_episode_length)]
        self.commands[ids, 0] = (ref[:, 18] + 2 * torch.rand(len(ids), device=self.device) - 1).clamp(0, 3)
        self.commands[ids, 1] = (ref[:, 19] + .2 * torch.rand(len(ids), device=self.device) - .1).clamp(-.1, .1)
        self.commands[ids, 2] = 3 * torch.rand(len(ids), device=self.device) - 1.5
        moving = torch.linalg.norm(self.commands[ids, :2], dim=1) > .1
        self.commands[ids, :2] *= moving[:, None]

    def reset_idx(self, ids):
        ids = torch.as_tensor(ids, device=self.device, dtype=torch.long).flatten()
        if ids.numel() == 0:
            return
        n = len(ids)
        self.qpos[ids] = 0
        self.qpos[ids, :3] = torch.tensor([0., 0., .35], device=self.device)
        self.qpos[ids, 3] = 1
        joint_randomization = (.5 + torch.rand(n, 12, device=self.device)) if self.domain_randomization else 1.0
        self.qpos[ids[:, None], torch.tensor(self.joint_qpos, device=self.device)] = (
            self.default_dof_pos * joint_randomization)
        self.qvel[ids] = 0
        if self.domain_randomization:
            self.qvel[ids, :6] = torch.rand(n, 6, device=self.device) - .5
        self.ctrl[ids] = 0
        self.motor_offsets[ids] = (-.035 + .07 * torch.rand(n, 12, device=self.device)
                                   if self.domain_randomization else 0.0)
        self.Kp_factors[ids] = (.9 + .2 * torch.rand(n, 1, device=self.device)
                                if self.domain_randomization else 1.0)
        self.Kd_factors[ids] = (.9 + .2 * torch.rand(n, 1, device=self.device)
                                if self.domain_randomization else 1.0)
        self.actions[ids] = 0
        self.last_actions[ids] = 0
        self.last_dof_vel[ids] = 0
        self.last_foot_velocities[ids] = 0
        self.last_contacts[ids] = False
        self.episode_length_buf[ids] = 0
        self.imitation_index[ids] = 0
        self._sample_commands(ids)
        mjw.forward(self.model, self.data)
        self._refresh()

    def reset(self, env_ids=None):
        if env_ids is None:
            env_ids = torch.arange(self.num_envs, device=self.device)
        self.reset_idx(env_ids)
        self._observations()
        return self.obs_buf, self.privileged_obs_buf

    def _observations(self):
        ref = self.motion[self.imitation_index.clamp(0, self.max_episode_length)]
        phase = (self.imitation_index.float() / self.motion.shape[0])[:, None]
        dof_offset = self.dof_pos - self.default_dof_pos
        common = (self.projected_gravity, self.commands[:, :3] * self.commands_scale,
                  dof_offset, self.dof_vel * .05, self.actions)
        self.obs_buf = torch.cat((self.base_ang_vel * .25, *common), dim=1)
        self.privileged_obs_buf = torch.cat((self.base_lin_vel * 2, self.base_ang_vel * .25,
            *common, phase, ref[:, 6:18], ref[:, 22:34], ref[:, 36:40]), dim=1)
        if self.obs_buf.shape[1] != 45 or self.privileged_obs_buf.shape[1] != 77:
            raise AssertionError("APEX observation shape drift")
        # Mirror the original 45-dimension noise layout, including its indexing.
        noise = torch.zeros(45, device=self.device)
        noise[:3] = .1 * 2
        noise[3:6] = .2 * .25
        noise[6:9] = .1
        noise[12:24] = .02
        noise[24:36] = 1.5 * .05
        if self.add_noise:
            self.obs_buf = self.obs_buf + (2 * torch.rand_like(self.obs_buf) - 1) * noise
        self.obs_buf = self.obs_buf.clamp(-100, 100)
        self.privileged_obs_buf = self.privileged_obs_buf.clamp(-100, 100)

    def _rewards(self):
        ref = self.motion[self.imitation_index.clamp(0, self.max_episode_length)]
        standing = torch.linalg.norm(self.commands[:, :2], dim=1) < .1
        joint_target = torch.where(standing[:, None], self.default_dof_pos, ref[:, 6:18])
        angle = torch.exp(-((self.dof_pos - joint_target).square().mean(dim=1)) / .01)
        foot_target = torch.where(standing[:, None], self.default_foot_pos_body_frame, ref[:, 22:34])
        foot_body = yaw_rotate_inverse(self.base_quat,
            self.foot_positions - self.base_pos[:, None, :]).reshape(self.num_envs, 12)
        foot = torch.exp(-(foot_target - foot_body).square().sum(dim=1) / .01)
        quat = torch.exp(-(ref[:, 36:40] - self.base_quat).square().sum(dim=1) / .5)
        group1 = 3.5 * angle + 2.5 * foot + .5 * quat

        lin = torch.exp(-(self.commands[:, :2] - self.base_lin_vel[:, :2]).square().sum(dim=1) / .25)
        ang = torch.exp(-(self.commands[:, 2] - self.base_ang_vel[:, 2]).square() / .25)
        collision = (torch.linalg.norm(self.contact_forces[:, self.penalty_ids], dim=-1) > .1).float().sum(dim=1)
        foot_contact = self.contact_forces[:, self.foot_ids, 2] > 1
        contact_filt = foot_contact | self.last_contacts
        slip = (contact_filt * self.foot_velocities[:, :, :2].square().sum(dim=2)).sum(dim=1)
        impact = -torch.minimum((self.foot_velocities[:, :, 2] - self.last_foot_velocities[:, :, 2]).square(),
                                torch.tensor(2., device=self.device)).sum(dim=1)
        height = (self.base_pos[:, 2] - ref[:, 21]).square()
        group2 = (2 * lin + 1.5 * ang - .00001 * self.torques.square().sum(dim=1)
                  - 2.5e-7 * ((self.last_dof_vel - self.dof_vel) / self.dt).square().sum(dim=1)
                  - collision - .01 * (self.last_actions - self.actions).square().sum(dim=1)
                  - .04 * slip + .0025 * impact - 10 * height
                  - .05 * self.base_ang_vel[:, :2].square().sum(dim=1))
        self.rew_buf = self.dt * torch.stack((group1, group2), dim=1)
        self.last_contacts = foot_contact
        self.last_foot_velocities = self.foot_velocities.clone()

    def step(self, actions):
        self.actions = actions.to(self.device).clamp(-100, 100)
        for _ in range(self.decimation):
            ref = self.motion[self.imitation_index.clamp(0, self.max_episode_length), 6:18]
            factor = (.99 ** (self.torque_ref_decay_factor / 100)) if self.enable_prior else 0.0
            self.decap_factor.fill_(factor)
            target = self.default_dof_pos + .25 * self.actions
            self.torques = (self.p_gains * self.Kp_factors * (target - self.dof_pos + self.motor_offsets)
                - self.d_gains * self.Kd_factors * self.dof_vel
                + factor * self.p_gains * (ref - self.dof_pos))
            self.torques = (self.torques * self.motor_strengths).clamp(-self.torque_limits, self.torque_limits)
            self.ctrl[:, self.motor_ids] = self.torques
            mjw.step(self.model, self.data)
        # step() integrates qpos after forward dynamics. Recompute derived body
        # poses and contact outputs for the newly integrated state.
        mjw.forward(self.model, self.data)
        self._refresh()
        self.episode_length_buf += 1
        self.common_step_counter += 1
        if self.resample_commands and self.common_step_counter % 250 == 0:
            self._sample_commands(torch.arange(self.num_envs, device=self.device))
        if self.push_robots and self.common_step_counter % 200 == 0:
            self.qvel[:, :2] = 1.2 * torch.rand(self.num_envs, 2, device=self.device) - .6
            self.qvel[:, 3:6] = 1.6 * torch.rand(self.num_envs, 3, device=self.device) - .8
            mjw.forward(self.model, self.data)
            self._refresh()
        self.time_out_buf = self.episode_length_buf > self.max_episode_length
        terminate = (torch.linalg.norm(self.contact_forces[:, self.termination_ids], dim=-1) > 1).any(dim=1)
        self.reset_buf = terminate | self.time_out_buf
        self._rewards()
        dones, timeouts = self.reset_buf.clone(), self.time_out_buf.clone()
        extras = {"time_outs": timeouts, "decap_factor": self.decap_factor[0].item()}
        # Rewards use reference k for the action just applied. The returned
        # observation must describe reference k+1 for the next policy action.
        self.imitation_index = (self.imitation_index + 1).clamp(max=self.max_episode_length)
        self.last_actions = self.actions.clone()
        self.last_dof_vel = self.dof_vel.clone()
        if self.auto_reset:
            self.reset_idx(dones.nonzero(as_tuple=False).flatten())
        self._observations()
        self.torque_ref_decay_factor += 1
        self.extras = extras
        return self.obs_buf, self.privileged_obs_buf, self.rew_buf, dones, extras

    def get_observations(self):
        return self.obs_buf

    def get_privileged_observations(self):
        return self.privileged_obs_buf


## apex_paths.py


In [ ]:
%%writefile apex_paths.py
"""Locate the existing APEX checkout used by the MuJoCo Warp port."""

from __future__ import annotations

import os
from pathlib import Path


def find_apex_root(explicit: str | Path | None = None) -> Path:
    candidates = []
    if explicit is not None:
        candidates.append(Path(explicit).expanduser())
    elif os.environ.get("APEX_ROOT"):
        candidates.append(Path(os.environ["APEX_ROOT"]).expanduser())
    else:
        here = Path(__file__).resolve().parent
        for base in (Path.cwd(), here, here.parent):
            candidates.extend((base / "APEX", base / "apex_mjwarp" / "APEX", base))
        candidates.append(Path("/home/hoge/university/laboratory/go2_2/apex_mjwarp/APEX"))

    for candidate in candidates:
        root = candidate.resolve()
        if ((root / "legged_gym/envs/param_config.yaml").is_file()
                and (root / "rsl_rl/rsl_rl/runners/on_policy_runner.py").is_file()):
            return root
    locations = ", ".join(str(p) for p in candidates)
    raise FileNotFoundError(
        "Existing APEX checkout was not found. Run inside apex_mjwarp, "
        "or pass --apex-root /path/to/apex_mjwarp/APEX. Searched: " + locations
    )


## snn_actor.py


In [ ]:
%%writefile snn_actor.py
"""Deterministic, stateless SNN Actor for APEX's unchanged Multi-Critic PPO.

The neuron states are reset for each policy decision and unrolled for a fixed
number of internal SNN ticks. This makes the action likelihood a function of
the stored observation alone, as required by APEX's feed-forward PPO storage.
"""

from __future__ import annotations

import torch
from torch import nn


def surrogate_spike(x: torch.Tensor, slope: float = 5.0) -> torch.Tensor:
    hard = (x > 0).to(x.dtype)
    soft = torch.sigmoid(slope * x)
    return hard + soft - soft.detach()


class SpikeActor(nn.Module):
    """Restored original SNN widths: 64 input and 256 output neurons/population."""

    def __init__(self, obs_dim: int, act_dim: int,
                 hidden_sizes: tuple[int, int] = (256, 256),
                 encoder_pop_dim: int = 64, decoder_pop_dim: int = 256,
                 snn_steps: int = 4):
        super().__init__()
        if min(*hidden_sizes, encoder_pop_dim, decoder_pop_dim, snn_steps) <= 0:
            raise ValueError("SNN sizes and tick count must be positive")
        self.obs_dim = obs_dim
        self.act_dim = act_dim
        self.hidden_sizes = hidden_sizes
        self.encoder_pop_dim = encoder_pop_dim
        self.decoder_pop_dim = decoder_pop_dim
        self.snn_steps = snn_steps

        input_pop = 2 * obs_dim * encoder_pop_dim
        output_pop = 2 * act_dim * decoder_pop_dim
        self.encoder_gain = nn.Parameter(torch.ones(obs_dim))
        self.encoder_bias = nn.Parameter(torch.zeros(obs_dim))
        thresholds = (torch.arange(encoder_pop_dim, dtype=torch.float32) + 0.5) / encoder_pop_dim
        self.register_buffer("encoder_thresholds", thresholds.reshape(1, 1, -1))

        self.Linear1 = nn.Linear(input_pop, hidden_sizes[0])
        self.Linear2 = nn.Linear(hidden_sizes[0], hidden_sizes[1])
        self.Linear3 = nn.Linear(hidden_sizes[1], output_pop)
        for layer in (self.Linear1, self.Linear2, self.Linear3):
            nn.init.kaiming_normal_(layer.weight)
            nn.init.zeros_(layer.bias)

        # A continuous membrane readout prevents a silent final spike layer
        # from forcing all deterministic motor commands to exactly zero.
        self.readout_norm = nn.LayerNorm(hidden_sizes[1])
        self.motor_readout = nn.Linear(hidden_sizes[1], act_dim)
        nn.init.orthogonal_(self.motor_readout.weight, gain=0.40)
        nn.init.zeros_(self.motor_readout.bias)
        self.spike_gain = nn.Parameter(torch.ones(act_dim))
        self.register_buffer("last_spike_rates", torch.zeros(3), persistent=False)

    def _encode(self, obs: torch.Tensor) -> torch.Tensor:
        x = torch.tanh(obs * self.encoder_gain + self.encoder_bias)
        probability = torch.cat((x.clamp_min(0), (-x).clamp_min(0)), dim=-1)
        return surrogate_spike(probability.unsqueeze(-1) - self.encoder_thresholds).flatten(1)

    @staticmethod
    def _lif(current: torch.Tensor, membrane: torch.Tensor,
             previous_spike: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        membrane = 0.5 * membrane + current - previous_spike
        return surrogate_spike(membrane - 1.0), membrane

    def forward(self, obs: torch.Tensor) -> torch.Tensor:
        if obs.shape[-1] != self.obs_dim:
            raise ValueError(f"Expected {self.obs_dim} Actor observations, got {obs.shape[-1]}")
        encoded = self._encode(obs)
        current1 = self.Linear1(encoded)
        batch = obs.shape[0]
        mem1 = obs.new_zeros(batch, self.hidden_sizes[0])
        mem2 = obs.new_zeros(batch, self.hidden_sizes[1])
        mem3 = obs.new_zeros(batch, 2 * self.act_dim * self.decoder_pop_dim)
        spk1 = torch.zeros_like(mem1)
        spk2 = torch.zeros_like(mem2)
        spk3 = torch.zeros_like(mem3)
        spike_counts = torch.zeros_like(mem3)
        rate_counts = obs.new_zeros(3)

        for _ in range(self.snn_steps):
            spk1, mem1 = self._lif(current1, mem1, spk1)
            spk2, mem2 = self._lif(self.Linear2(spk1), mem2, spk2)
            spk3, mem3 = self._lif(self.Linear3(spk2), mem3, spk3)
            spike_counts = spike_counts + spk3
            rate_counts = rate_counts + torch.stack((spk1.mean(), spk2.mean(), spk3.mean()))

        population_rate = (spike_counts / self.snn_steps).reshape(batch, 2 * self.act_dim,
                                                                   self.decoder_pop_dim).mean(-1)
        signed_rate = population_rate[:, :self.act_dim] - population_rate[:, self.act_dim:]
        membrane_action = 2.0 * torch.tanh(self.motor_readout(self.readout_norm(mem2)))
        mean = membrane_action + 0.25 * self.spike_gain * signed_rate
        self.last_spike_rates = (rate_counts / self.snn_steps).detach()
        return mean


def build_actor_critic_class(official_class):
    """Replace only the Actor module; retain APEX's critics and distribution API."""

    class SNNMultiCriticActorCritic(official_class):
        is_recurrent = False

        def __init__(self, num_actor_obs, num_critic_obs, num_actions,
                     snn_hidden_sizes=(256, 256), encoder_pop_dim=64,
                     decoder_pop_dim=256, snn_steps=4, **kwargs):
            super().__init__(num_actor_obs, num_critic_obs, num_actions, **kwargs)
            self.actor = SpikeActor(num_actor_obs, num_actions,
                                    hidden_sizes=tuple(snn_hidden_sizes),
                                    encoder_pop_dim=encoder_pop_dim,
                                    decoder_pop_dim=decoder_pop_dim,
                                    snn_steps=snn_steps)

    SNNMultiCriticActorCritic.__name__ = "SNNMultiCriticActorCritic"
    return SNNMultiCriticActorCritic


## train_snn.py


In [ ]:
%%writefile train_snn.py
"""Train a Go2 SNN Actor using APEX's unchanged Multi-Critic PPO runner."""

from __future__ import annotations

import argparse
import importlib.util
import json
import os
from pathlib import Path
import sys

import torch

from env_snn import ApexGo2Warp
from apex_paths import find_apex_root
from snn_actor import build_actor_critic_class


def load_official_runner(apex_root: Path):
    sys.path.insert(0, str(apex_root / "rsl_rl"))
    from rsl_rl.modules import MultiCriticActorCritic

    runner_path = apex_root / "rsl_rl/rsl_rl/runners/on_policy_runner.py"
    spec = importlib.util.spec_from_file_location("apex_snn_on_policy_runner", runner_path)
    if spec is None or spec.loader is None:
        raise RuntimeError(f"Cannot load APEX runner: {runner_path}")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    module.SNNMultiCriticActorCritic = build_actor_critic_class(MultiCriticActorCritic)
    return module.OnPolicyRunner


def make_runner(apex_root: Path, num_envs: int, log_dir: Path,
                motion: str, snn_steps: int):
    apex_root = apex_root.resolve()
    if not (apex_root / "legged_gym/envs/param_config.yaml").is_file():
        raise FileNotFoundError(f"Not an APEX repository: {apex_root}")
    if not (apex_root / motion).is_file():
        raise FileNotFoundError(f"Motion CSV not found: {apex_root / motion}")
    log_dir = log_dir.resolve()
    log_dir.mkdir(parents=True, exist_ok=True)
    os.environ.setdefault("WANDB_MODE", "offline")
    torch.manual_seed(39)

    # The APEX storage and runner read this relative configuration file.
    os.chdir(apex_root)
    OnPolicyRunner = load_official_runner(apex_root)
    env = ApexGo2Warp(apex_root, num_envs=num_envs, motion=motion)
    cfg = {
        "policy": {
            "num_critics": 2,
            "actor_hidden_dims": [512, 256, 128],
            "critic_hidden_dims": [512, 256, 128],
            "activation": "elu",
            "init_noise_std": 1.0,
            "snn_hidden_sizes": [256, 256],
            "encoder_pop_dim": 64,
            "decoder_pop_dim": 256,
            "snn_steps": snn_steps,
        },
        "algorithm": {
            "value_loss_coef": 1.0,
            "use_clipped_value_loss": True,
            "clip_param": 0.2,
            "entropy_coef": 0.01,
            "num_learning_epochs": 5,
            "num_mini_batches": 4,
            "learning_rate": 1e-3,
            "schedule": "adaptive",
            "gamma": 0.99,
            "lam": 0.95,
            "desired_kl": 0.01,
            "max_grad_norm": 1.0,
        },
        "runner": {
            "policy_class_name": "SNNMultiCriticActorCritic",
            "algorithm_class_name": "MultiCriticPPO",
            "num_steps_per_env": 24,
            "save_interval": 200,
            "experiment_name": "apex_go2_snn_mjwarp",
            "run_name": "paper_snn_actor_64x256",
            "critic_num": 2,
        },
    }
    (log_dir / "apex_snn_run.json").write_text(json.dumps({
        "apex_root": str(apex_root),
        "motion": motion,
        "num_envs": num_envs,
        "snn_hidden_sizes": [256, 256],
        "encoder_pop_dim": 64,
        "decoder_pop_dim": 256,
        "snn_steps": snn_steps,
        "actor_obs": 45,
        "critic_obs": 77,
        "algorithm": "official MultiCriticPPO",
    }, indent=2), encoding="utf-8")
    runner = OnPolicyRunner(env, cfg, log_dir=str(log_dir), device=str(env.device))
    return env, runner


def check_actor_path(env, runner):
    actor = runner.alg.actor_critic.actor
    obs = env.get_observations()[:4]
    mean = actor(obs)
    head_grad, first_grad = torch.autograd.grad(
        mean.square().mean(),
        (actor.motor_readout.weight, actor.Linear1.weight),
        allow_unused=False,
    )
    mean_abs = float(mean.detach().abs().mean())
    head_grad_abs = float(head_grad.detach().abs().mean())
    first_grad_abs = float(first_grad.detach().abs().mean())
    print(f"SNN Actor preflight: |mean|={mean_abs:.4f}, "
          f"head_grad={head_grad_abs:.2e}, first_grad={first_grad_abs:.2e}, "
          f"spike_rates={actor.last_spike_rates.tolist()}")
    if not torch.isfinite(mean).all() or mean_abs < 0.03:
        raise RuntimeError("SNN motor mean is zero or non-finite before PPO")
    if head_grad_abs < 1e-10 or first_grad_abs < 1e-12:
        raise RuntimeError("PPO gradient does not reach the SNN motor readout and first layer")


def check_prior_motion(apex_root: Path, motion: str) -> bool:
    """Measure prior-only behavior without assuming a policy-free gait must survive."""
    probe = ApexGo2Warp(apex_root, num_envs=1, motion=motion,
                        add_noise=False, enable_prior=True,
                        domain_randomization=False, resample_commands=False,
                        push_robots=False, auto_reset=False)
    probe.commands[0, :3] = torch.tensor([1.0, 0.0, 0.0], device=probe.device)
    probe._observations()
    zeros = torch.zeros(1, 12, device=probe.device)
    speeds = []
    survived = 0
    with torch.no_grad():
        for _ in range(200):
            _, _, _, done, _ = probe.step(zeros)
            speeds.append(float(probe.base_lin_vel[0, 0]))
            survived += 1
            if bool(done[0]):
                break
    mean_speed = sum(speeds) / len(speeds)
    print(f"Prior-only preflight: survived={survived}/200, "
          f"mean vx={mean_speed:+.3f} m/s, peak vx={max(speeds):+.3f} m/s")
    success = survived >= 100 and max(speeds) >= 0.10
    if not success:
        print("Prior-only probe did not sustain locomotion. APEX's prior also "
              "needs policy actions; inspect the Warp contact/PD behavior before a long run.")
    del probe
    return success


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--apex-root", type=Path, default=None,
                        help="Existing apex_mjwarp/APEX checkout; found automatically when omitted")
    parser.add_argument("--num-envs", type=int, default=1024,
                        help="4096 reproduces the official environment count; 1024 uses less GPU memory")
    parser.add_argument("--iterations", type=int, default=1200)
    parser.add_argument("--log-dir", type=Path, default=Path("runs/apex_go2_snn_paper"))
    parser.add_argument("--motion", default="imitation_data/animal_mocap/go2_retarget_canter_2ms.csv")
    parser.add_argument("--snn-steps", type=int, default=4)
    parser.add_argument("--resume", type=Path, default=None)
    parser.add_argument("--smoke", action="store_true", help="Run one PPO update with 64 worlds")
    parser.add_argument("--require-prior-motion", action="store_true",
                        help="Abort if the independent prior-only probe fails")
    args = parser.parse_args()
    if args.num_envs <= 0 or args.iterations <= 0:
        parser.error("--num-envs and --iterations must be positive")
    if args.smoke:
        args.num_envs, args.iterations = 64, 1

    apex_root = find_apex_root(args.apex_root)
    print(f"Using existing APEX checkout: {apex_root}")
    prior_ok = check_prior_motion(apex_root, args.motion)
    if args.require_prior_motion and not prior_ok:
        raise RuntimeError("Prior-only motion requirement failed")
    env, runner = make_runner(apex_root, args.num_envs, args.log_dir,
                              args.motion, args.snn_steps)
    if args.resume is not None:
        runner.load(str(args.resume.resolve()))
        env.torque_ref_decay_factor = runner.current_learning_iteration * runner.num_steps_per_env
        env.common_step_counter = env.torque_ref_decay_factor
        print(f"Resumed at iteration {runner.current_learning_iteration}; "
              f"DecAP step={env.torque_ref_decay_factor}")
    check_actor_path(env, runner)
    print(f"APEX Multi-Critic PPO: worlds={args.num_envs}, "
          f"iterations={args.iterations}, rollout=24, "
          f"SNN={runner.alg.actor_critic.actor.hidden_sizes}, "
          f"encoder=64, decoder=256, ticks={args.snn_steps}")
    runner.learn(num_learning_iterations=args.iterations, init_at_random_ep_len=False)


if __name__ == "__main__":
    main()


## evaluate_snn.py


In [ ]:
%%writefile evaluate_snn.py
"""Evaluate a trained SNN APEX Actor with Action Prior disabled."""

from __future__ import annotations

import argparse
import csv
import json
import os
from pathlib import Path
import sys

import mujoco
import torch

from env_snn import ApexGo2Warp
from apex_paths import find_apex_root
from snn_actor import build_actor_critic_class


def load_policy(apex_root: Path, checkpoint: Path, snn_steps: int, device: str):
    os.chdir(apex_root)
    sys.path.insert(0, str(apex_root / "rsl_rl"))
    from rsl_rl.modules import MultiCriticActorCritic

    cls = build_actor_critic_class(MultiCriticActorCritic)
    policy = cls(45, 77, 12, num_critics=2,
                 actor_hidden_dims=[512, 256, 128],
                 critic_hidden_dims=[512, 256, 128],
                 activation="elu", init_noise_std=1.0,
                 snn_hidden_sizes=(256, 256), encoder_pop_dim=64,
                 decoder_pop_dim=256, snn_steps=snn_steps).to(device)
    saved = torch.load(checkpoint, map_location=device)
    policy.load_state_dict(saved["model_state_dict"])
    policy.eval()
    return policy, saved.get("iter")


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--apex-root", default=None, type=Path,
                        help="Existing apex_mjwarp/APEX checkout; found automatically when omitted")
    parser.add_argument("--checkpoint", required=True, type=Path)
    parser.add_argument("--output", type=Path, default=Path("apex_snn_prior_off_eval.csv"))
    parser.add_argument("--video", type=Path, default=None)
    parser.add_argument("--vx", type=float, default=1.0,
                        help="Official canter training commands use positive forward speeds")
    parser.add_argument("--vy", type=float, default=0.0)
    parser.add_argument("--wz", type=float, default=0.0)
    parser.add_argument("--steps", type=int, default=500)
    parser.add_argument("--snn-steps", type=int, default=None)
    parser.add_argument("--motion", default=None)
    args = parser.parse_args()
    apex_root = find_apex_root(args.apex_root)
    checkpoint = args.checkpoint.resolve()
    metadata_path = checkpoint.parent / "apex_snn_run.json"
    metadata = json.loads(metadata_path.read_text(encoding="utf-8")) if metadata_path.is_file() else {}
    snn_steps = args.snn_steps or metadata.get("snn_steps", 4)
    motion = args.motion or metadata.get("motion", "imitation_data/animal_mocap/go2_retarget_canter_2ms.csv")
    if args.snn_steps is not None and metadata and args.snn_steps != metadata.get("snn_steps"):
        parser.error("--snn-steps differs from the checkpoint's run metadata")
    output = args.output.resolve()
    video = args.video.resolve() if args.video else None
    output.parent.mkdir(parents=True, exist_ok=True)
    if video:
        video.parent.mkdir(parents=True, exist_ok=True)
    if args.steps <= 0:
        parser.error("--steps must be positive")

    policy, iteration = load_policy(apex_root, checkpoint, snn_steps, "cuda:0")
    env = ApexGo2Warp(apex_root, num_envs=1, motion=motion,
                      add_noise=False, enable_prior=False,
                      domain_randomization=False, resample_commands=False,
                      push_robots=False, auto_reset=False)
    env.reset()
    env.commands[0, :3] = torch.tensor([args.vx, args.vy, args.wz], device=env.device)
    env._observations()

    image_writer = None
    renderer = None
    render_data = None
    if video:
        import imageio.v2 as imageio

        image_writer = imageio.get_writer(video, fps=round(1.0 / env.dt))
        renderer = mujoco.Renderer(env.cpu_model, width=640, height=480)
        render_data = mujoco.MjData(env.cpu_model)

    rows = []
    try:
        for step_no in range(args.steps):
            with torch.no_grad():
                action = policy.act_inference(env.get_observations())
                _, _, rewards, dones, _ = env.step(action)
            row = {
                "step": step_no,
                "time_s": (step_no + 1) * env.dt,
                "command_vx": args.vx,
                "command_vy": args.vy,
                "command_wz": args.wz,
                "vx_body": float(env.base_lin_vel[0, 0]),
                "vy_body": float(env.base_lin_vel[0, 1]),
                "wz_body": float(env.base_ang_vel[0, 2]),
                "mean_action_abs": float(action[0].abs().mean()),
                "mean_torque_abs": float(env.torques[0].abs().mean()),
                "reward_style": float(rewards[0, 0]),
                "reward_task": float(rewards[0, 1]),
                "done": bool(dones[0]),
            }
            rows.append(row)
            if image_writer:
                render_data.qpos[:] = env.qpos[0].detach().cpu().numpy()
                render_data.qvel[:] = env.qvel[0].detach().cpu().numpy()
                mujoco.mj_forward(env.cpu_model, render_data)
                renderer.update_scene(render_data)
                image_writer.append_data(renderer.render())
            if bool(dones[0]):
                break
    finally:
        if image_writer:
            image_writer.close()
        if renderer:
            renderer.close()

    with output.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
    n = len(rows)
    avg = lambda key: sum(row[key] for row in rows) / n
    print(f"Prior OFF: checkpoint iter={iteration}, steps={n}/{args.steps}, "
          f"vx={avg('vx_body'):+.3f}, vy={avg('vy_body'):+.3f} m/s, "
          f"wz={avg('wz_body'):+.3f} rad/s, "
          f"|action|={avg('mean_action_abs'):.3f}, "
          f"|torque|={avg('mean_torque_abs'):.2f} Nm")
    print(f"Trace: {output}")
    if video:
        print(f"Video: {video}")


if __name__ == "__main__":
    main()


## probe_csv_prior_cpu.py


In [ ]:
%%writefile probe_csv_prior_cpu.py
"""CPU MuJoCo check of official canter CSV and additive APEX joint prior."""

from __future__ import annotations

import argparse
import csv
from pathlib import Path
import xml.etree.ElementTree as ET

import mujoco
import numpy as np

from go2_mjcf import build_go2_mjcf
from apex_paths import find_apex_root


JOINTS = tuple(f"{leg}_{part}_joint" for leg in ("FL", "FR", "RL", "RR")
               for part in ("hip", "thigh", "calf"))
DEFAULT = np.array([0.1, 0.8, -1.5, -0.1, 0.8, -1.5,
                    0.1, 1.0, -1.5, -0.1, 1.0, -1.5])


def run(apex_root: Path, steps: int = 200, scene: Path | None = None,
        enable_prior: bool = True) -> dict:
    urdf = apex_root / "resources/robots/go2/urdf/go2.urdf"
    motion_path = apex_root / "imitation_data/animal_mocap/go2_retarget_canter_2ms.csv"
    model = (mujoco.MjModel.from_xml_path(str(scene)) if scene is not None
             else mujoco.MjModel.from_xml_string(build_go2_mjcf(urdf)))
    data = mujoco.MjData(model)
    with motion_path.open(newline="", encoding="utf-8") as handle:
        csv_rows = list(csv.reader(handle))
    motion = np.asarray(csv_rows[1:], dtype=np.float64)
    q_idx = np.array([model.jnt_qposadr[model.joint(name).id] for name in JOINTS])
    v_idx = np.array([model.jnt_dofadr[model.joint(name).id] for name in JOINTS])
    a_idx = np.array([model.actuator(name if scene is None else name.removesuffix("_joint")).id
                      for name in JOINTS])
    urdf_root = ET.parse(urdf).getroot()
    effort = {joint.get("name"): float(joint.find("limit").get("effort"))
              for joint in urdf_root.findall("joint") if joint.get("type") == "revolute"}
    limit = np.array([effort[name] for name in JOINTS])
    data.qpos[2] = 0.35
    data.qpos[3] = 1.0
    data.qpos[q_idx] = DEFAULT
    mujoco.mj_forward(model, data)
    vx, heights = [], []
    for control_step in range(steps):
        ref = motion[min(control_step, len(motion) - 1), 6:18]
        prior = (0.99 ** (control_step / 100)) if enable_prior else 0.0
        for _ in range(4):
            q = data.qpos[q_idx]
            dq = data.qvel[v_idx]
            torque = 20.0 * (DEFAULT - q) - 0.5 * dq + prior * 20.0 * (ref - q)
            data.ctrl[a_idx] = np.clip(torque, -limit, limit)
            mujoco.mj_step(model, data)
        vx.append(float(data.qvel[0]))
        heights.append(float(data.qpos[2]))
        if data.qpos[2] < 0.18:
            break
    result = {"steps": len(vx), "mean_vx": float(np.mean(vx)),
              "peak_vx": float(np.max(vx)), "final_x": float(data.qpos[0]),
              "min_height": float(np.min(heights))}
    print(result)
    return result


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--apex-root", type=Path, default=None)
    parser.add_argument("--steps", type=int, default=200)
    parser.add_argument("--scene", type=Path, default=None)
    parser.add_argument("--without-prior", action="store_true")
    args = parser.parse_args()
    run(find_apex_root(args.apex_root), args.steps,
        args.scene.resolve() if args.scene is not None else None,
        enable_prior=not args.without_prior)


## CPU上でAction Priorの物理挙動を確認

MuJoCo Warp本学習前に変換MJCFと公式モーションCSVを調べます。Prior単体で転倒しても、Actorを含むAPEX学習が不可能という証明ではありません。


In [ ]:
subprocess.run([sys.executable, 'probe_csv_prior_cpu.py', '--apex-root', str(APEX_ROOT)], check=True)


## 最小スモークテスト

64環境でPPOを1回更新します。Actor出力と第1層までの勾配も検査します。


In [ ]:
subprocess.run([sys.executable, 'train_snn.py', '--apex-root', str(APEX_ROOT), '--smoke', '--log-dir', 'runs/apex_snn_smoke'], check=True)


## 本学習

公式Go2設定は4096環境・1200 iterationです。GPUメモリが足りない場合は`--num-envs 1024`で実行できます。下の `RUN_TRAIN` をTrueにしてください。


In [ ]:
RUN_TRAIN = False
if RUN_TRAIN:
    subprocess.run([sys.executable, 'train_snn.py', '--apex-root', str(APEX_ROOT), '--num-envs', '4096', '--iterations', '1200', '--log-dir', 'runs/apex_snn_full'], check=True)


## Prior OFFで評価

学習済みチェックポイントを指定します。公式canter設定の学習内指令は前進です。後退・大きい横移動はこの単一モーション設定の学習範囲外です。


In [ ]:
CHECKPOINT = Path('runs/apex_snn_full/model_1200.pt')
if CHECKPOINT.is_file():
    subprocess.run([sys.executable, 'evaluate_snn.py', '--apex-root', str(APEX_ROOT), '--checkpoint', str(CHECKPOINT), '--vx', '1.0', '--vy', '0.0', '--wz', '0.0', '--output', 'apex_snn_forward_trace.csv', '--video', 'apex_snn_forward.mp4'], check=True)
else:
    print('Checkpoint not found yet:', CHECKPOINT)
